In [36]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, chi2
from scipy.stats import pearsonr
import io
import re

In [22]:
# 1. Read the file as text
with open('/content/Loan-Approval.csv', 'r') as file:
    lines = file.readlines()

# 2. Clean the lines by removing the surrounding quotes
cleaned_lines = []
for line in lines:
    # Remove the quotes at the beginning and end of each line
    cleaned_line = line.strip().strip('"')
    cleaned_lines.append(cleaned_line)

# 3. Create a new StringIO object to parse the CSV line by line
string_data = io.StringIO('\n'.join(cleaned_lines))

# 4. Read header
header = next(csv.reader(io.StringIO(cleaned_lines[0])))
num_columns = len(header)
print(f"Expected number of columns: {num_columns}")

# 5. Process the data line by line to handle inconsistencies
data = []
problem_lines = []

# Create a CSV reader
csv_reader = csv.reader(io.StringIO('\n'.join(cleaned_lines[1:])))

for i, row in enumerate(csv_reader, start=1):
    if len(row) == num_columns:
        data.append(row)
    else:
        problem_lines.append((i+1, len(row), row))  # +1 for header, +1 for 0-indexing
        # Try to fix it - if too many columns, merge text fields that might have been split incorrectly
        if len(row) > num_columns:
            # For example, one strategy could be to merge excess columns
            fixed_row = row[:num_columns-1] + [','.join(row[num_columns-1:])]
            if len(fixed_row) == num_columns:
                data.append(fixed_row)
                print(f"Fixed line {i+1} by merging columns")

# Show information about problematic lines
if problem_lines:
    print(f"\nFound {len(problem_lines)} lines with incorrect number of fields:")
    for line_num, num_fields, content in problem_lines[:5]:  # Show up to 5 problem lines
        print(f"Line {line_num}: Has {num_fields} fields instead of {num_columns}")
        print(f"Content: {content[:5]}...{content[-5:] if len(content) > 10 else content[5:]}")

# 6. Create a DataFrame from the processed data
df = pd.DataFrame(data, columns=header)

# 7. Verify the data is now properly formatted
print("\nFixed DataFrame shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())

# 8. Save the fixed CSV for further processing
df.to_csv('/content/Fixed-Loan-Approval.csv', index=False)
print("\nFixed CSV saved to '/content/Fixed-Loan-Approval.csv'")

# Now we can continue with the preprocessing tasks

Expected number of columns: 14
Fixed line 256 by merging columns
Fixed line 488 by merging columns
Fixed line 496 by merging columns

Found 10 lines with incorrect number of fields:
Line 74: Has 13 fields instead of 14
Content: ['LP001248', 'Male', 'No', '0', 'Graduate']...['81', '300', '1', 'Semiurban', 'Y']
Line 81: Has 13 fields instead of 14
Content: ['LP001264', 'Male', 'Yes', '3+', 'Not Graduate']...['130', '360', '', 'Semiurban', 'Y']
Line 88: Has 13 fields instead of 14
Content: ['LP001280', 'Male', 'Yes', '2', 'Not Graduate']...['99', '360', '', 'Semiurban', 'Y']
Line 117: Has 13 fields instead of 14
Content: ['LP001401', 'Male', 'Yes', '1', 'Graduate']...['185', '180', '1', 'Rural', 'Y']
Line 189: Has 13 fields instead of 14
Content: ['LP001643', 'Male', 'Yes', '0', 'Graduate']...['58', '360', '', 'Rural', 'Y']

Fixed DataFrame shape: (607, 14)

First 5 rows:
    Loan_ID Gender Married Dependents     Education Self_Employed  \
0  LP001002   Male      No          0      Gradua

In [28]:
# 1. Load the dataset
# Replace 'Loan-Approval.csv' with the actual file path
df = pd.read_csv('/content/Fixed-Loan-Approval.csv')
print("Dataset loaded successfully.")
print("\nInitial shape:", df.shape)
print("\nFirst 5 rows:\n")
df.head()

Dataset loaded successfully.

Initial shape: (607, 14)

First 5 rows:



,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Risk,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,No,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,No,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,No,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,No,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,No,Urban,Y


In [29]:
# 2. Display an overview of the data
print("\n### Data Overview ###")
print("\nData Info (structure and types):")
print(df.info())
print("\nFirst 5 rows:")
print(df.head())
print("\nMissing values per column:")
print(df.isnull().sum())
print("\nShape of the dataframe:", df.shape)
print("\nInitial summary statistics:")
print(df.describe())


### Data Overview ###

Data Info (structure and types):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 607 entries, 0 to 606
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            607 non-null    object 
 1   Gender             593 non-null    object 
 2   Married            604 non-null    object 
 3   Dependents         592 non-null    object 
 4   Education          607 non-null    object 
 5   Self_Employed      575 non-null    object 
 6   ApplicantIncome    607 non-null    int64  
 7   CoapplicantIncome  607 non-null    float64
 8   LoanAmount         585 non-null    float64
 9   Loan_Amount_Term   594 non-null    float64
 10  Credit_History     560 non-null    float64
 11  Risk               592 non-null    object 
 12  Property_Area      607 non-null    object 
 13  Loan_Status        607 non-null    object 
dtypes: float64(4), int64(1), object(9)
memory usage: 66.5+ KB
None

F

In [30]:
# 3. Preprocessing Steps

# 3.1 Handle incorrect entries
# Assumption: Incorrect entries include negative values in numeric columns where they don't make sense
# (e.g., ApplicantIncome, LoanAmount) or unexpected categories in categorical columns.
numeric_cols = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History']
categorical_cols = ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Risk', 'Property_Area', 'Loan_Status']

# Check for negative values in numeric columns (except Credit_History, which can be 0 or 1)
for col in numeric_cols[:-1]:  # Exclude Credit_History
    if (df[col] < 0).any():
        print(f"\nNegative values found in {col}. Replacing with NaN.")
        df[col] = df[col].mask(df[col] < 0, np.nan)

# Check Credit_History for values other than 0 or 1
if not df['Credit_History'].isin([0, 1, np.nan]).all():
    print("\nInvalid values in Credit_History. Replacing with NaN.")
    df['Credit_History'] = df['Credit_History'].mask(~df['Credit_History'].isin([0, 1]), np.nan)

# Check unique values in categorical columns for unexpected entries
for col in categorical_cols:
    print(f"\nUnique values in {col}: {df[col].unique()}")


Unique values in Gender: ['Male' 'Female' nan]

Unique values in Married: ['No' 'Yes' nan]

Unique values in Dependents: ['0' '1' '2' '3+' nan]

Unique values in Education: ['Graduate' 'Not Graduate']

Unique values in Self_Employed: ['No' 'Yes' nan]

Unique values in Risk: ['No' 'Yes' nan 'Nio']

Unique values in Property_Area: ['Urban' 'Rural' 'Semiurban' 'No']

Unique values in Loan_Status: ['Y' 'N' 'Urban,N' 'Rural,N']


In [31]:
# 3.2 Handle missing data
# Solution 1: Using Pandas
print("\n### Handling Missing Data with Pandas ###")
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())
for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])
print("Missing values after Pandas imputation:")
print(df.isnull().sum())


### Handling Missing Data with Pandas ###
Missing values after Pandas imputation:
Loan_ID              0
Gender               0
Married              0
Dependents           0
Education            0
Self_Employed        0
ApplicantIncome      0
CoapplicantIncome    0
LoanAmount           0
Loan_Amount_Term     0
Credit_History       0
Risk                 0
Property_Area        0
Loan_Status          0
dtype: int64


In [32]:
# 3.3 Categorical label encoding
print("\n### Encoding Categorical Variables ###")
# Drop Loan_ID as it’s an identifier and not useful for modeling
df = df.drop('Loan_ID', axis=1)

# Update categorical_cols after dropping Loan_ID
categorical_cols = [col for col in categorical_cols if col in df.columns]

# Encode binary categorical columns
binary_cols = ['Gender', 'Married', 'Education', 'Self_Employed', 'Risk', 'Loan_Status']
le = LabelEncoder()
for col in binary_cols:
    df[col] = le.fit_transform(df[col])
    print(f"{col} encoded: {df[col].unique()}")

# Encode ordinal variable Dependents
dependents_mapping = {'0': 0, '1': 1, '2': 2, '3+': 3}
df['Dependents'] = df['Dependents'].map(dependents_mapping)
print(f"Dependents encoded: {df['Dependents'].unique()}")

# One-hot encode nominal variable Property_Area
df = pd.get_dummies(df, columns=['Property_Area'], prefix='Property')
print("\nColumns after one-hot encoding Property_Area:", df.columns.tolist())


### Encoding Categorical Variables ###
Gender encoded: [1 0]
Married encoded: [0 1]
Education encoded: [0 1]
Self_Employed encoded: [0 1]
Risk encoded: [1 2 0]
Loan_Status encoded: [3 0 2 1]
Dependents encoded: [0 1 2 3]

Columns after one-hot encoding Property_Area: ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History', 'Risk', 'Loan_Status', 'Property_No', 'Property_Rural', 'Property_Semiurban', 'Property_Urban']


In [33]:
# 3.4 Outlier detection and treatment
print("\n### Handling Outliers ###")
outlier_cols = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term']
def cap_outliers(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return series.clip(lower_bound, upper_bound)

for col in outlier_cols:
    df[col] = cap_outliers(df[col])
    print(f"{col} outliers capped.")


### Handling Outliers ###
ApplicantIncome outliers capped.
CoapplicantIncome outliers capped.
LoanAmount outliers capped.
Loan_Amount_Term outliers capped.


In [34]:
# 3.5 Transformation (e.g., log transformation for skewed data)
print("\n### Applying Transformations ###")
for col in outlier_cols:
    skewness = df[col].skew()
    print(f"Skewness of {col}: {skewness:.3f}")
    if skewness > 1:
        df[col] = np.log1p(df[col])
        print(f"Applied log1p transformation to {col}")


### Applying Transformations ###
Skewness of ApplicantIncome: 1.035
Applied log1p transformation to ApplicantIncome
Skewness of CoapplicantIncome: 1.017
Applied log1p transformation to CoapplicantIncome
Skewness of LoanAmount: 0.669
Skewness of Loan_Amount_Term: 0.000


In [35]:
# 3.7 Column data type conversion
print("\n### Converting Column Data Types ###")
# Ensure all columns are numeric after encoding
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')
print("Data types after conversion:")
print(df.dtypes)


### Converting Column Data Types ###
Data types after conversion:
Gender                  int64
Married                 int64
Dependents              int64
Education               int64
Self_Employed           int64
ApplicantIncome       float64
CoapplicantIncome     float64
LoanAmount            float64
Loan_Amount_Term      float64
Credit_History        float64
Risk                    int64
Loan_Status             int64
Property_No              bool
Property_Rural           bool
Property_Semiurban       bool
Property_Urban           bool
dtype: object


In [44]:
# 3.8 Hot-fix for True/False columns
print("\n### Hot-Fix: Converting True/False Columns ###")
# Identify boolean columns (e.g., from get_dummies)
for col in df.columns:
    if df[col].dtype == bool or df[col].isin([True, False]).any():
        print(f"Converting {col} from True/False to 1/0")
        df[col] = df[col].astype(int)
print("Columns after hot-fix:")
print(df.dtypes)


### Hot-Fix: Converting True/False Columns ###
Converting Gender from True/False to 1/0
Converting Married from True/False to 1/0
Converting Dependents from True/False to 1/0
Converting Education from True/False to 1/0
Converting Self_Employed from True/False to 1/0
Converting CoapplicantIncome from True/False to 1/0
Converting Credit_History from True/False to 1/0
Converting Risk from True/False to 1/0
Converting Loan_Status from True/False to 1/0
Converting Property_No from True/False to 1/0
Converting Property_Rural from True/False to 1/0
Converting Property_Semiurban from True/False to 1/0
Converting Property_Urban from True/False to 1/0
Columns after hot-fix:
Gender                  int64
Married                 int64
Dependents              int64
Education               int64
Self_Employed           int64
ApplicantIncome       float64
CoapplicantIncome       int64
LoanAmount            float64
Loan_Amount_Term      float64
Credit_History          int64
Risk                    in

In [38]:
# 4. Split the data into input features (X) and target variable (y)
print("\n### Splitting Data into X and y ###")
X = df.drop('Loan_Status', axis=1)
y = df['Loan_Status']
print("X shape:", X.shape)
print("y shape:", y.shape)

# Apply standardization to X
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)
print("\nX after standardization (first 5 rows):")
print(X_scaled.head())


### Splitting Data into X and y ###
X shape: (607, 15)
y shape: (607,)

X after standardization (first 5 rows):
     Gender   Married  Dependents  Education  Self_Employed  ApplicantIncome  \
0  0.473065 -1.369955   -0.735681  -0.522050      -0.392419         0.677912   
1  0.473065  0.729951    0.256671  -0.522050      -0.392419         0.232866   
2  0.473065  0.729951   -0.735681  -0.522050       2.548299        -0.540225   
3  0.473065  0.729951   -0.735681   1.915524      -0.392419        -0.813249   
4  0.473065 -1.369955   -0.735681  -0.522050      -0.392419         0.724420   

   CoapplicantIncome  LoanAmount  Loan_Amount_Term  Credit_History      Risk  \
0          -1.105492   -0.171270               0.0        0.414505 -0.420063   
1           0.793854   -0.171270               0.0        0.414505 -0.420063   
2          -1.105492   -1.285024               0.0        0.414505 -0.420063   
3           0.909796   -0.314980               0.0        0.414505 -0.420063   
4     

In [39]:
# 5. Feature Selection
# 5.1 Pearson correlation for numerical variables
print("\n### Feature Selection: Pearson Correlation ###")
for col in outlier_cols:
    corr, p_value = pearsonr(X[col], y)  # Using X (pre-scaled) since correlation is scale-invariant
    print(f"{col}: correlation={corr:.3f}, p-value={p_value:.3f}")


### Feature Selection: Pearson Correlation ###
ApplicantIncome: correlation=0.012, p-value=0.765
CoapplicantIncome: correlation=0.060, p-value=0.140
LoanAmount: correlation=-0.044, p-value=0.280
Loan_Amount_Term: correlation=nan, p-value=nan


<ipython-input-39-dc070164b80c>:5: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  corr, p_value = pearsonr(X[col], y)  # Using X (pre-scaled) since correlation is scale-invariant


In [40]:
# 5.2 Chi-squared test for categorical variables
print("\n### Feature Selection: Chi-Squared Test ###")
# Update categorical features after encoding
cat_features = ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Risk',
                'Property_Rural', 'Property_Semiurban', 'Property_Urban']
selector = SelectKBest(chi2, k='all')
selector.fit(X[cat_features], y)
scores = selector.scores_
p_values = selector.pvalues_
for i, feature in enumerate(cat_features):
    print(f"{feature}: score={scores[i]:.3f}, p-value={p_values[i]:.3f}")


### Feature Selection: Chi-Squared Test ###
Gender: score=0.172, p-value=0.982
Married: score=2.314, p-value=0.510
Dependents: score=0.972, p-value=0.808
Education: score=5.254, p-value=0.154
Self_Employed: score=5.899, p-value=0.117
Risk: score=21.230, p-value=0.000
Property_Rural: score=4.871, p-value=0.181
Property_Semiurban: score=7.705, p-value=0.053
Property_Urban: score=1.760, p-value=0.624


In [41]:
# 6. Summary outputs before and after transformations
print("\n### Summary Statistics After Preprocessing ###")
print(X_scaled.describe())


### Summary Statistics After Preprocessing ###
             Gender       Married    Dependents     Education  Self_Employed  \
count  6.070000e+02  6.070000e+02  6.070000e+02  6.070000e+02   6.070000e+02   
mean  -5.852906e-18  7.316132e-17  1.170581e-17  1.755872e-17   5.560260e-17   
std    1.000825e+00  1.000825e+00  1.000825e+00  1.000825e+00   1.000825e+00   
min   -2.113875e+00 -1.369955e+00 -7.356812e-01 -5.220505e-01  -3.924186e-01   
25%    4.730648e-01 -1.369955e+00 -7.356812e-01 -5.220505e-01  -3.924186e-01   
50%    4.730648e-01  7.299509e-01 -7.356812e-01 -5.220505e-01  -3.924186e-01   
75%    4.730648e-01  7.299509e-01  2.566710e-01 -5.220505e-01  -3.924186e-01   
max    4.730648e-01  7.299509e-01  2.241375e+00  1.915524e+00   2.548299e+00   

       ApplicantIncome  CoapplicantIncome    LoanAmount  Loan_Amount_Term  \
count     6.070000e+02       6.070000e+02  6.070000e+02             607.0   
mean      9.379281e-16      -1.141317e-16 -1.024258e-16               0.0   


In [45]:
df.head()

,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Risk,Loan_Status,Property_No,Property_Rural,Property_Semiurban,Property_Urban
0,1,0,0,0,0,8.674197,0,128.0,360.0,1,1,3,0,0,0,1
1,1,1,1,0,0,8.430327,7,128.0,360.0,1,1,0,0,1,0,0
2,1,1,0,0,1,8.006701,0,66.0,360.0,1,1,3,0,0,0,1
3,1,1,0,1,0,7.857094,7,120.0,360.0,1,1,3,0,0,0,1
4,1,0,0,0,0,8.699681,0,141.0,360.0,1,1,3,0,0,0,1
